In [1]:
!pip install -q torchcontrib

import os
import sys
import torch
import pandas as pd
import numpy as np
import subprocess

from tqdm.auto import tqdm

  Preparing metadata (setup.py) ... done


In [2]:
BASE = "/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021"

DF_PARTS = [
    os.path.join(BASE, "ASVspoof2021_DF_eval_part00", "ASVspoof2021_DF_eval"),
    os.path.join(BASE, "ASVspoof2021_DF_eval_part01", "ASVspoof2021_DF_eval"),
    os.path.join(BASE, "ASVspoof2021_DF_eval_part02", "ASVspoof2021_DF_eval"),
]

METADATA = os.path.join(
    BASE,
    "DF-keys-full",
    "keys",
    "DF",
    "CM",
    "trial_metadata.txt"
)

print("Dataset found:", os.path.exists(BASE))
print("Metadata found:", os.path.exists(METADATA))

Dataset found: True
Metadata found: True


In [3]:
df2021 = pd.read_csv(
    METADATA,
    sep=r"\s+",
    header=None
)

df2021 = df2021[[1, 5]]
df2021.columns = ["audio_id", "label"]

df2021["label"] = df2021["label"].map({
    "spoof": 0,
    "bonafide": 1
})

print("Total samples:", len(df2021))
print(df2021["label"].value_counts())

Total samples: 611829
label
0    589212
1     22617
Name: count, dtype: int64


In [4]:
audio_lookup = {}

for part in DF_PARTS:
    flac_dir = os.path.join(part, "flac")

    for filename in os.listdir(flac_dir):
        if filename.endswith(".flac"):
            audio_id = filename[:-5]
            audio_lookup[audio_id] = os.path.join(flac_dir, filename)

df2021["path"] = df2021["audio_id"].map(audio_lookup)

print("Audio files found:", len(audio_lookup))
print("Missing files:", df2021["path"].isna().sum())

Audio files found: 458871
Missing files: 152958


In [5]:
df2021_eval = df2021.dropna(subset=["path"]).reset_index(drop=True)

print("Evaluation samples:", len(df2021_eval))
print(df2021_eval["label"].value_counts())

Evaluation samples: 458871
label
0    441894
1     16977
Name: count, dtype: int64


In [6]:
# Create a balanced evaluation subset

spoof_df = df2021_eval[df2021_eval["label"] == 0]
bonafide_df = df2021_eval[df2021_eval["label"] == 1]

# Keep all bona fide samples and randomly select the same number of spoof samples
spoof_sample = spoof_df.sample(
    n=len(bonafide_df),
    random_state=42
)

# Combine them and shuffle
df2021_balanced = pd.concat(
    [spoof_sample, bonafide_df],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Balanced evaluation samples:", len(df2021_balanced))
print(df2021_balanced["label"].value_counts())

Balanced evaluation samples: 33954
label
1    16977
0    16977
Name: count, dtype: int64


In [7]:
def load_audio_ffmpeg(path):
    cmd = [
        "ffmpeg",
        "-v", "error",
        "-i", path,
        "-f", "f32le",
        "-ac", "1",
        "-ar", "16000",
        "-"
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=True
    )

    return np.frombuffer(
        result.stdout,
        dtype=np.float32
    ).copy()

In [8]:
class DF2021Dataset(torch.utils.data.Dataset):

    def __init__(self, df, nb_samp=64600):
        self.df = df
        self.nb_samp = nb_samp

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        wav = load_audio_ffmpeg(row["path"])
        wav = torch.from_numpy(wav)

        # Pad / crop to 64600 samples
        if len(wav) < self.nb_samp:
            wav = torch.nn.functional.pad(
                wav,
                (0, self.nb_samp - len(wav))
            )
        else:
            wav = wav[:self.nb_samp]

        label = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return wav, label

In [9]:
df2021_dataset = DF2021Dataset(df2021_balanced)

df2021_loader = torch.utils.data.DataLoader(
    df2021_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Samples:", len(df2021_dataset))
print("Batches:", len(df2021_loader))

Samples: 33954
Batches: 1062


In [10]:
!git clone https://github.com/clovaai/aasist.git /kaggle/working/aasist

Cloning into '/kaggle/working/aasist'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 38 (delta 12), reused 6 (delta 6), pack-reused 10 (from 1)
Receiving objects: 100% (38/38), 1.43 MiB | 18.49 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [11]:
import importlib.util
import torch

aasist_path = "/kaggle/working/aasist/models/AASIST.py"

spec = importlib.util.spec_from_file_location("AASIST", aasist_path)
aasist = importlib.util.module_from_spec(spec)
spec.loader.exec_module(aasist)

Model = aasist.Model

model_config = {
    "architecture": "AASIST",
    "nb_samp": 64600,
    "first_conv": 128,
    "filts": [70, [1,32], [32,32], [32,64], [64,64]],
    "gat_dims": [64,32],
    "pool_ratios": [0.5,0.7,0.5,0.5],
    "temperatures": [2.0,2.0,100.0,100.0]
}

model = Model(model_config).cuda()

checkpoint = torch.load(
    "/kaggle/working/aasist/models/weights/AASIST.pth",
    map_location="cuda"
)

model.load_state_dict(checkpoint)
model.eval()

print("AASIST loaded successfully!")

AASIST loaded successfully!


# Inference cell : Evaluation/Testing


In [12]:
all_labels = []
all_scores = []

with torch.no_grad():

    for wav, labels in tqdm(df2021_loader):

        wav = wav.cuda(non_blocking=True)

        output = model(wav)

        # AASIST returns (output, features)
        scores = output[0][:, 1]

        all_labels.extend(labels.numpy())
        all_scores.extend(scores.cpu().numpy())

print("Inference complete!")
print("Samples processed:", len(all_labels))

  0%|          | 0/1062 [00:00<?, ?it/s]

Inference complete!
Samples processed: 33954


# Below is the EER calculation cell


    -> [5% EER  → excellent] [10% EER → good] [20% EER → weaker]

    -> We are not taking accuracy as the main metric coz our data set has:
    
            Spoof:     441,894
            Bonafide:   16,977
        this suggests - 96.3% spoof
        Thus a poor classifier that predicts - 
            everything = spoof,
        will have an accuracy of 96.3%.

In [13]:
from sklearn.metrics import roc_curve
import numpy as np

labels = np.array(all_labels)
scores = np.array(all_scores)

# ROC curve
fpr, tpr, thresholds = roc_curve(labels, scores)

# False Negative Rate
fnr = 1 - tpr

# Find the point where FPR and FNR are closest
eer_index = np.nanargmin(np.abs(fpr - fnr))

eer = (fpr[eer_index] + fnr[eer_index]) / 2

print(f"EER: {eer * 100:.2f}%")#

EER: 19.49%


In [14]:
from sklearn.metrics import accuracy_score

labels = np.array(all_labels)
scores = np.array(all_scores)

# AASIST score: higher = more likely bonafide
predictions = (scores >= 0.5).astype(int)

accuracy = accuracy_score(labels, predictions)

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 67.09%


In [15]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, predictions)

print("Confusion Matrix:")
print(cm)

print(f"\nSpoof predicted correctly:    {cm[0,0]}")
print(f"Spoof predicted as bonafide: {cm[0,1]}")
print(f"Bonafide predicted correctly: {cm[1,1]}")
print(f"Bonafide predicted as spoof:  {cm[1,0]}")

Confusion Matrix:
[[ 6567 10410]
 [  764 16213]]

Spoof predicted correctly:    6567
Spoof predicted as bonafide: 10410
Bonafide predicted correctly: 16213
Bonafide predicted as spoof:  764


In [16]:
import os

TRAIN_FLAC_DIR = "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/flac"

print("Directory exists:", os.path.exists(TRAIN_FLAC_DIR))
print("Directory:", TRAIN_FLAC_DIR)

Directory exists: True
Directory: /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/flac


In [17]:
import os

PROTOCOL_DIR = "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_asv_protocols"

print(os.listdir(PROTOCOL_DIR))

['ASVspoof2019.LA.asv.eval.male.trn.txt', 'ASVspoof2019.LA.asv.eval.female.trn.txt', 'ASVspoof2019.LA.asv.dev.female.trn.txt', 'ASVspoof2019.LA.asv.dev.female.trl.txt', 'ASVspoof2019.LA.asv.eval.gi.trl.txt', 'ASVspoof2019.LA.asv.dev.gi.trl.txt', 'ASVspoof2019.LA.asv.dev.male.trl.txt', 'ASVspoof2019.LA.asv.dev.male.trn.txt', 'ASVspoof2019.LA.asv.eval.female.trl.txt', 'ASVspoof2019.LA.asv.eval.male.trl.txt']


In [18]:
CM_PROTOCOL_DIR = "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols"

print(os.listdir(CM_PROTOCOL_DIR))

['ASVspoof2019.LA.cm.eval.trl.txt', 'ASVspoof2019.LA.cm.train.trn.txt', 'ASVspoof2019.LA.cm.dev.trl.txt']


In [19]:
import pandas as pd

TRAIN_PROTOCOL = os.path.join(
    CM_PROTOCOL_DIR,
    "ASVspoof2019.LA.cm.train.trn.txt"
)

df2019_train = pd.read_csv(
    TRAIN_PROTOCOL,
    sep=r"\s+",
    header=None
)

print("Shape:", df2019_train.shape)
print("\nFirst 5 rows:")
print(df2019_train.head())

Shape: (25380, 5)

First 5 rows:
         0             1  2  3         4
0  LA_0079  LA_T_1138215  -  -  bonafide
1  LA_0079  LA_T_1271820  -  -  bonafide
2  LA_0079  LA_T_1272637  -  -  bonafide
3  LA_0079  LA_T_1276960  -  -  bonafide
4  LA_0079  LA_T_1341447  -  -  bonafide


In [20]:
df2019_train = df2019_train[[1, 4]]
df2019_train.columns = ["audio_id", "label"]

df2019_train["label"] = df2019_train["label"].map({
    "spoof": 0,
    "bonafide": 1
})

df2019_train["path"] = df2019_train["audio_id"].apply(
    lambda x: os.path.join(TRAIN_FLAC_DIR, x + ".flac")
)

print("Training samples:", len(df2019_train))
print("\nClass distribution:")
print(df2019_train["label"].value_counts())

print("\nFirst 5 rows:")
print(df2019_train.head())

Training samples: 25380

Class distribution:
label
0    22800
1     2580
Name: count, dtype: int64

First 5 rows:
       audio_id  label                                               path
0  LA_T_1138215      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...
1  LA_T_1271820      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...
2  LA_T_1272637      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...
3  LA_T_1276960      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...
4  LA_T_1341447      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...


In [21]:
test_path = df2019_train.iloc[0]["path"]

print("Test file:")
print(test_path)
print("\nExists:", os.path.exists(test_path))

Test file:
/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/flac/LA_T_1138215.flac

Exists: True


In [22]:
DEV_DIR = "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_dev"
DEV_FLAC_DIR = os.path.join(DEV_DIR, "flac")

DEV_PROTOCOL = os.path.join(
    CM_PROTOCOL_DIR,
    "ASVspoof2019.LA.cm.dev.trl.txt"
)

df2019_dev = pd.read_csv(
    DEV_PROTOCOL,
    sep=r"\s+",
    header=None
)

print("Dev shape:", df2019_dev.shape)
print("\nFirst 5 rows:")
print(df2019_dev.head())

Dev shape: (24844, 5)

First 5 rows:
         0             1  2  3         4
0  LA_0069  LA_D_1047731  -  -  bonafide
1  LA_0069  LA_D_1105538  -  -  bonafide
2  LA_0069  LA_D_1125976  -  -  bonafide
3  LA_0069  LA_D_1293230  -  -  bonafide
4  LA_0069  LA_D_1340209  -  -  bonafide


In [23]:
# Keep only audio ID and label
df2019_dev = df2019_dev[[1, 4]]
df2019_dev.columns = ["audio_id", "label"]

# Convert labels to numbers
df2019_dev["label"] = df2019_dev["label"].map({
    "spoof": 0,
    "bonafide": 1
})

# Add audio paths
df2019_dev["path"] = df2019_dev["audio_id"].apply(
    lambda x: os.path.join(DEV_FLAC_DIR, x + ".flac")
)

print("Development samples:", len(df2019_dev))
print("\nClass distribution:")
print(df2019_dev["label"].value_counts())

print("\nFirst 5 rows:")
print(df2019_dev.head())

Development samples: 24844

Class distribution:
label
0    22296
1     2548
Name: count, dtype: int64

First 5 rows:
       audio_id  label                                               path
0  LA_D_1047731      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...
1  LA_D_1105538      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...
2  LA_D_1125976      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...
3  LA_D_1293230      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...
4  LA_D_1340209      1  /kaggle/input/datasets/awsaf49/asvpoof-2019-da...


In [24]:
from torch.utils.data import DataLoader

# Reuse the same Dataset class you used for ASVspoof 2021
train_dataset = DF2021Dataset(df2019_train)
dev_dataset = DF2021Dataset(df2019_dev)

train_loader = DataLoader(
    train_dataset,
    batch_size=24,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=24,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Training samples:", len(train_dataset))
print("Training batches:", len(train_loader))

print("\nDevelopment samples:", len(dev_dataset))
print("Development batches:", len(dev_loader))

Training samples: 25380
Training batches: 1058

Development samples: 24844
Development batches: 1036


In [25]:
import torch.nn as nn

# Class-weighted loss to handle the 2019 train imbalance
class_weights = torch.tensor([0.1, 0.9], device="cuda")
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Small learning rate for fine-tuning
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

print("Optimizer and loss configured!")
print("Learning rate:", optimizer.param_groups[0]["lr"])

Optimizer and loss configured!
Learning rate: 1e-05
